# FloodValue: знакомство с исходным GeoTIFF

Этот вводный ноутбук помогает проверить среду и прочитать метаданные выбранного S1-файла. Условия работы заданы выданными «Постановка_кейса_3.docx» и «Критерии_кейса_3.docx»; [GitHub-справочник](https://github.com/eugenetatarchenkolaw/test_EO) дополняет их.

Обязательный источник — [официальный Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11). Файлы, независимое разделение и демонстрационный чип выбирает команда. Код и модели/веса собственного решения публикуются в **GitVerse**.

Ноутбук не скачивает весь набор и не создает прогнозы, портфель или заказы.

## 1. Подготовить среду

Подойдет Python 3.12 с Rasterio. При отсутствии библиотеки установите ее в среде ноутбука командой `%pip install rasterio==1.5.1`. После установки при необходимости перезапустите ядро.

При работе из копии справочника можно установить `requirements.txt`; зависимости обучения собственного метода выбирает команда.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path
import rasterio

print("Python:", sys.version.split()[0])
print("Rasterio:", version("rasterio"))

## 2. Выбрать официальный S1-файл

Получите выбранный GeoTIFF из источника авторов и укажите локальный путь ниже. В Colab сначала загрузите файл в свою сессию. Пустой путь оставляет просмотр метаданных пропущенным: это не проверка снимка.

Сохраните исходный URL, версию или контрольную сумму, chip_id/event_id, дату, лицензию и роль файла в `source_manifest.json`. Ручная QC-метка должна относиться к тому же чипу; не у всех файлов есть ручная разметка.

In [ ]:
S1_PATH = ""  # Локальный путь к самостоятельно полученному официальному S1 GeoTIFF.

## 3. Посмотреть метаданные

В описании Sen1Floods11 S1 имеет два канала VV/VH в dB и размер 512 × 512. Сверьте описание с фактическим файлом: CRS, transform, nodata и тип данных важны для последующего результата. Значение `nodata=None` не доказывает отсутствие пропусков.

Этот просмотр не проверяет происхождение файла, пригодность пикселей или независимость выборки. [Rasterio Quickstart](https://rasterio.readthedocs.io/en/stable/quickstart.html) объясняет чтение GeoTIFF подробнее.

In [ ]:
if not S1_PATH:
    print("Просмотр пропущен: задайте S1_PATH для выбранного официального S1-файла.")
else:
    source = Path(S1_PATH).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f"Файл не найден: {source}")
    with rasterio.open(source) as dataset:
        print("Файл:", source.name)
        print("Размер (строки, столбцы):", dataset.shape)
        print("Каналы:", dataset.count, dataset.descriptions)
        print("Типы данных:", dataset.dtypes)
        print("CRS:", dataset.crs)
        print("Transform:", dataset.transform)
        print("Nodata по каналам:", dataset.nodatavals)

## 4. Подготовить собственный эксперимент

В официальной ручной метке −1 означает невалидную разметку, 0 — не воду, 1 — воду. Метка 1 сама по себе не выделяет временное затопление. До эксперимента определите целевой класс, split и порядок настройки без использования итогового test.

По постановке нужны пороговый baseline и самостоятельный основной метод. Их сравнивают на идентичных независимых валидных пикселях по IoU, F1/Dice, Precision, Recall и числу пикселей. Обязательные вероятности, неопределенность и экономика относятся к основному методу. Обеспечьте S1-only и хотя бы один контролируемый эксперимент; при дополнительных источниках сравните его с расширенным вариантом.

Для размещения десяти точек используются валидная область исходного S1 и фиксированный seed, без карт прогноза, ущерба и ручных тестовых меток. Заданные V/q доступны в [таблице типов](https://github.com/eugenetatarchenkolaw/test_EO/blob/main/data/asset_types.csv). Собственные зоны и экономический расчет выполняются в проекте команды.

[Учебные ресурсы](https://github.com/eugenetatarchenkolaw/test_EO/blob/main/docs/ML_GUIDE.md) · [Проверка](https://github.com/eugenetatarchenkolaw/test_EO/blob/main/docs/VALIDATION.md) · [Форматы результатов](https://github.com/eugenetatarchenkolaw/test_EO/blob/main/docs/CONTRACTS.md).

В GitVerse сохраните весь код, конфигурации и используемые модели/веса с инструкциями обучения, inference на другом официальном S1 и пересчета бюджета. Успешный просмотр метаданных не является выполнением кейса.